# 06 — Named Entity Recognition
### IndicNews AI — Hindi News Analysis, Retrieval & Recommendation System

Extracts Person / Location / Organization entities via Stanza's Hindi
pipeline.



In [6]:

import stanza
stanza.download("hi")  

2026-07-10 12:16:08 INFO: Downloaded file to C:\Users\Admin\AppData\Local\StanfordNLP\stanza\Cache\1.13.0\resources\resources.json
2026-07-10 12:16:08 INFO: Downloading default packages for language: hi (Hindi) ...
2026-07-10 12:16:09 INFO: File exists: C:\Users\Admin\AppData\Local\StanfordNLP\stanza\Cache\1.13.0\resources\hi\default.zip
2026-07-10 12:16:13 INFO: Finished downloading models and saved to C:\Users\Admin\AppData\Local\StanfordNLP\stanza\Cache\1.13.0\resources


[['zip', 'default.zip']]

In [7]:
import sys
sys.path.append("..")

import pandas as pd
from data_utils import load_dataset
from ner import get_pipeline, extract_entities, group_entities_by_type, TAG_MAP

pd.set_option("display.max_colwidth", 80)
nlp = get_pipeline()
print("Pipeline loaded.")

Pipeline loaded.


## 1. NER on a single real headline

Sanity check before running at any scale.

In [8]:
df = load_dataset(verbose=False)
sample_headline = df["Headline"].iloc[0]
print("Text:", sample_headline)

entities = extract_entities(sample_headline, nlp)
for ent in entities:
    print(f"  {ent['text']:20s} raw={ent['raw_type']:10s} mapped={ent['type']}")

Text: कांग्रेस नेता बलजिंदर सिंह की पंजाब में घर के अंदर गोली मारकर की गई हत्या
  बलजिंदर सिंह         raw=NEP        mapped=Person
  पंजाब                raw=NEL        mapped=Location
  कांग्रेस             raw=GAZETTEER  mapped=Organization


## 2. Raw tag inventory — verify `TAG_MAP` against real output

Run NER across a sample of articles and list every distinct raw tag
Stanza actually produced, with counts. 

In [9]:
from collections import Counter

sample_df = df.drop_duplicates(subset=["Headline", "Content"]).sample(200, random_state=42)

raw_tag_counts = Counter()
for text in sample_df["Headline"]:
    for ent in extract_entities(text, nlp):
        raw_tag_counts[ent["raw_type"]] += 1

print("Raw tag counts (from 200 headlines):")
for tag, count in raw_tag_counts.most_common():
    if tag == "GAZETTEER":
        mapped = "Organization (gazetteer fallback, already final)"
    else:
        mapped = TAG_MAP.get(tag.upper(), "UNMAPPED (passthrough)")
    print(f"  {tag:15s} count={count:4d}  -> {mapped}")

Raw tag counts (from 200 headlines):
  NEL             count= 128  -> Location
  NEP             count= 126  -> Person
  NEN             count=  62  -> Number
  NETI            count=  31  -> Time
  NEO             count=  22  -> Organization
  GAZETTEER       count=  13  -> Organization (gazetteer fallback, already final)
  NEAR            count=   1  -> UNMAPPED (passthrough)


## 3. Full entity extraction demo — grouped by type

On a handful of real headlines, entities grouped into
Person/Location/Organization.

In [10]:
for text in sample_df["Headline"].head(5):
    entities = extract_entities(text, nlp)
    grouped = group_entities_by_type(entities)
    print(text)
    print(" ->", grouped)
    print()

पूर्व विदेश मंत्री नटवर सिंह की बिगड़ी तबीयत, गुरुग्राम के मेदांता अस्पताल में भर्ती
 -> {'Person': ['नटवर सिंह'], 'Location': ['गुरुग्राम'], 'Organization': ['मेदांता अस्पताल']}

पूर्ण सूर्य ग्रहण के दौरान Aditya-L1 रखेगा सूरज पर करीबी नजर
 -> {}

Pit bull: 8 साल की लड़की पर नोएडा में पिटबुल ने किया हमला, बच्ची बुरी तरह से हुई घायल.
 -> {'Number': ['8'], 'Location': ['नोएडा']}

बिहार जेडीयू के उपाध्यक्ष ने पार्टी से दिया इस्तीफा, कहा- आरजेडी से गठबंधन के बाद अपराध बढ़ा
 -> {'Organization': ['बिहार जेडीयू']}

ऐस्ट्रोफोटोग्राफर एंड्रयू मैकार्थी ने खींची चंद्रमा के दक्षिणी छोर की तस्वीर
 -> {'Person': ['एंड्रयू मैकार्थी']}



## 4. Timing check — is full-corpus NER feasible?

Stanza's neural pipeline is much slower than the regex/TF-IDF work in
earlier modules. Time a small batch and extrapolate before deciding
whether to run NER over the full 34,826-article corpus or just a
representative sample for this project's demo purposes.

In [ ]:
import time

timing_sample = sample_df["Content"].head(50)
t0 = time.time()
for text in timing_sample:
    extract_entities(text, nlp)
elapsed = time.time() - t0

print(f"{len(timing_sample)} articles (Content) took {elapsed:.1f}s "
      f"-> {elapsed/len(timing_sample)*1000:.0f} ms/article")
print(f"Estimated full corpus (34,826 articles): {elapsed/len(timing_sample)*34826/60:.1f} minutes")

**Real result: 493 ms/article, ~286 minutes (4.8 hours) extrapolated for
the full 34,826-article corpus.** Too slow to precompute over the full
training corpus. **Decision: NER runs on-demand per user submission in
the Module 10 Flask API**, not precomputed/cached for the training
corpus — a single article at ~0.5s is completely fine for an interactive
API response, and this matches NER's actual role here: it's a per-
submission inference feature (like the eventual similarity search
query), not a training input the way TF-IDF/embeddings were.

## 5. Summary

- **Tag scheme confirmed:** Stanza's Hindi NER uses FIRE2013 tags, not
  generic PERSON/LOC/ORG. `TAG_MAP` fixed and verified against real
  counts from 200 headlines: `NEL` 128→Location, `NEP` 126→Person,
  `NEN` 62→Number, `NETI` 31→Time, `NEO` 22→Organization. One rare tag
  (`NEAR`, 1 occurrence) left unmapped rather than guessed at.
- **Quality check, real examples:** `नटवर सिंह`→Person, `गुरुग्राम`→
  Location, `मेदांता अस्पताल`→Organization, `बिहार जेडीयू`→Organization
  (confirms political parties fall under Organization, not a separate
  category) — all correct.
- **Gazetteer fallback added and confirmed working:** Stanza's model
  missed `कांग्रेस` (Congress) in a real test sentence — a normal
  statistical-NER recall limitation, not a bug. Added a small known-party
  gazetteer (19 major parties, Devanagari + Roman script, case-
  preserved) merged into `extract_entities()`. Re-tested: `कांग्रेस`
  now correctly caught (13 gazetteer matches across the 200-headline
  sample). This is a safety net for well-known names, not a complete
  fix — regional parties/spelling variants outside the list still rely
  on the model alone.
- **Output is unfiltered by default**, per request — Person, Location,
  Organization, Number, Time all returned; `filter_core_entities()`
  remains available but optional if the three-type-only view is ever
  needed.
- **Timing: 493 ms/article → ~4.8 hours for the full corpus.** Too slow
  to precompute. **Final decision: NER runs on-demand in the Module 10
  API**, not cached over the training set.

Next: **07_TopicModeling.ipynb** — LDA/NMF topic modeling on the TF-IDF
features from Module 4.